# Partie 1

In [35]:
import pandas as pd
import numpy as np
import unicodedata
import re
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

# Étape 1 – Charger le fichier CSV
df = pd.read_csv("employes_classification_formatif.csv")
print("Étape 1 – Aperçu du fichier chargé :")
df.head()


Étape 1 – Aperçu du fichier chargé :


,ID,Nom,Âge,Salaire,Date Embauche,Note performance,Durée contrat,Départ
0,121,David,56,35 795€,11-Jul-17,5,douze mois,0
1,15,JACQUES,60,96820$,12/4/2019,2,douze mois,1
2,57,Benjamin,36,57194euros,2019.03.14,3,365 jours,0
3,204,Emmanuel,53,80263 €,9/15/2020,4,1 an,0
4,330,Cécile,39,84820 €,2016.11.24,5,365 jours,0


In [36]:
# Étape 2 – Supprimer les doublons complets
before_count = len(df)
# Masque des doublons (lignes identiques sur toutes les colonnes)
dup_mask = df.duplicated(keep='first')
dup_count = int(dup_mask.sum())

if dup_count > 0:
    print(f"\nÉtape 2 – Doublons détectés : {dup_count}")
    print("Aperçu des premières lignes dupliquées :")
    df[dup_mask].head()
else:
    print("\nÉtape 2 – Aucun doublon complet détecté.")

# Suppression des doublons complets
df = df.drop_duplicates(keep='first')
after_count = len(df)
print(f"Nombre de lignes après suppression des doublons : {after_count} (−{before_count - after_count})")


Étape 2 – Aucun doublon complet détecté.
Nombre de lignes après suppression des doublons : 420 (−0)


In [37]:
# Étape 3 – Uniformiser les noms de colonnes (robuste + résolution des collisions)
import keyword

def normalize_colname(col: str) -> str:
    col = str(col).lower().strip()
    # Supprimer les accents (diacritiques)
    col = ''.join(c for c in unicodedata.normalize('NFD', col) if unicodedata.category(c) != 'Mn')
    # Remplacer tout caractère non alphanumérique par _
    col = re.sub(r'[^0-9a-z]+', '_', col)
    # Réduire les multiples _ et retirer _ en début/fin
    col = re.sub(r'_+', '_', col).strip('_')
    # S'assurer que le nom n'est pas vide
    if not col:
        col = 'col'
    # Préfixer si commence par un chiffre
    if re.match(r'^\d', col):
        col = 'col_' + col
    # Éviter les mots-clés Python
    if keyword.iskeyword(col):
        col = col + '_'
    return col

original_cols = df.columns.tolist()
normalized_cols = [normalize_colname(c) for c in original_cols]

# Résoudre les collisions (noms duplicés après normalisation)
seen = {}
final_cols = []
for name in normalized_cols:
    if name not in seen:
        seen[name] = 0
        final_cols.append(name)
    else:
        seen[name] += 1
        final_cols.append(f"{name}_{seen[name]}")

# Appliquer
df.columns = final_cols

# Afficher un mapping clair
mapping = pd.DataFrame({
    'original': original_cols,
    'normalized': final_cols
})
print("\nÉtape 3 – Noms de colonnes après normalisation (avec collisions résolues) :")
print(mapping)

# Alerte si des collisions ont été résolues
collisions = mapping.groupby('normalized').size()
if (collisions > 1).any():
    dup_names = collisions[collisions > 1].index.tolist()
    print(f"\nAttention: collisions résolues pour les noms: {dup_names}")


Étape 3 – Noms de colonnes après normalisation (avec collisions résolues) :
           original        normalized
0                ID                id
1              Nom                nom
2               Âge               age
3           Salaire           salaire
4     Date Embauche     date_embauche
5  Note performance  note_performance
6     Durée contrat     duree_contrat
7            Départ            depart


In [39]:
# Étape 4 – Harmoniser les valeurs de la colonne nom (robuste + rapport)

def clean_name(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    # Retirer les accents
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    # Remplacer tout ce qui n'est pas lettre ou espace par un espace
    s = re.sub(r'[^a-z\s]', ' ', s)
    # Réduire les espaces multiples
    s = re.sub(r'\s+', ' ', s).strip()
    # Vider → valeur manquante
    return s if s else pd.NA
# Preserve name raw
if 'nom_raw' not in df.columns:
    df['nom_raw'] = df['nom']
before_names = df['nom'].copy()
df['nom'] = df['nom'].apply(clean_name)

# Rapport de nettoyage
total = len(df)
changed = (before_names.fillna('') != df['nom'].fillna('')).sum()
missing_before = before_names.isna().sum()
missing_after = df['nom'].isna().sum()
unique_before = before_names.astype(str).nunique(dropna=True)
unique_after = df['nom'].astype(str).nunique(dropna=True)

print("\nÉtape 4 – Harmonisation de 'nom' terminée :")
print(f"  • Lignes totales: {total}")
print(f"  • Valeurs modifiées: {changed}")
print(f"  • Manquants avant: {missing_before} → après: {missing_after}")
print(f"  • Valeurs uniques avant: {unique_before} → après: {unique_after}")

# Exemples avant/après
examples = pd.DataFrame({
    'avant': before_names.head(10),
    'apres': df['nom'].head(10)
})
print("\nAperçu des 10 premières transformations:")
print(examples)

# Détecter les collisions (différents 'avant' → même 'apres')
map_df = pd.DataFrame({'avant': before_names, 'apres': df['nom']}).dropna()
collisions = map_df.groupby('apres')['avant'].nunique().sort_values(ascending=False)
potential_collisions = collisions[collisions > 1].head(10)
if not potential_collisions.empty:
    print("\nAttention – Plusieurs noms originaux mappés sur le même nom nettoyé (top 10):")
    print(potential_collisions)
else:
    print("\nAucune collision notable détectée dans les 10 premiers regroupements.")



Étape 4 – Harmonisation de 'nom' terminée :
  • Lignes totales: 420
  • Valeurs modifiées: 407
  • Manquants avant: 0 → après: 0
  • Valeurs uniques avant: 260 → après: 178

Aperçu des 10 premières transformations:
       avant     apres
0     David      david
1    JACQUES   jacques
2   Benjamin  benjamin
3  Emmanuel   emmanuel
4     Cécile    cecile
5       Noël      noel
6   Hortense  hortense
7     Marcel    marcel
8       Anne      anne
9      Lucas     lucas

Attention – Plusieurs noms originaux mappés sur le même nom nettoyé (top 10):
apres
cecile        3
anouk         3
christiane    3
mathilde      3
juliette      3
jean          3
jerome        3
franck        3
valentine     3
sabine        3
Name: avant, dtype: int64


In [40]:
# Étape 5 – Corriger les types de données (incl. âges en texte → int)

# Sauvegarder la colonne originale
if 'age_raw' not in df.columns:
    df['age_raw'] = df['age']

def fr_text_to_int(x):
    import unicodedata, re
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    # Si contient des chiffres, extraire le premier entier
    m = re.search(r"\d+", s)
    if m:
        try:
            return int(m.group())
        except Exception:
            pass
    # Normaliser accents et nettoyer
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    s = re.sub(r"[\s_]+", "-", s)  # espaces → tirets
    s = s.replace("-et-", "-")      # retirer 'et'
    s = re.sub(r"-?ans?$", "", s)   # retirer 'an/ans'
    s = s.strip('-')
    if not s:
        return pd.NA

    # Dictionnaires
    units = {
        'zero':0,'un':1,'deux':2,'trois':3,'quatre':4,'cinq':5,
        'six':6,'sept':7,'huit':8,'neuf':9
    }
    teens = {
        'dix':10,'onze':11,'douze':12,'treize':13,'quatorze':14,
        'quinze':15,'seize':16,'dix-sept':17,'dix-huit':18,'dix-neuf':19
    }
    tens = {
        'vingt':20,'trente':30,'quarante':40,'cinquante':50,'soixante':60
    }
    specials = {
        'soixante-dix':70,
        'quatre-vingt':80,
        'quatre-vingts':80,
        'quatre-vingt-dix':90
    }

    # Direct mapping
    if s in units:
        return units[s]
    if s in teens:
        return teens[s]
    if s in tens:
        return tens[s]
    if s in specials:
        return specials[s]

    tokens = [t for t in s.split('-') if t]
    if not tokens:
        return pd.NA

    try:
        # Cas: tens + unit (vingt-deux, trente-un, etc.)
        if tokens[0] in tens:
            val = tens[tokens[0]]
            rest = tokens[1:]
            if not rest:
                return val
            rest_str = '-'.join(rest)
            if rest_str in units:
                return val + units[rest_str]
            if rest_str in teens:
                return val + teens[rest_str]
            # ex: "vingt-un" déjà couvert; sinon NA
            return pd.NA

        # Cas: soixante + (unit|teens|dix-[unit]) → 60 + X
        if tokens[0] == 'soixante':
            rest_str = '-'.join(tokens[1:])
            if rest_str in units:
                return 60 + units[rest_str]
            if rest_str in teens:
                return 60 + teens[rest_str]
            return pd.NA

        # Cas: quatre-vingt(s) + (unit|teens|dix|dix-[unit])
        if tokens[0] in ['quatre', 'quatre-vingt', 'quatre-vingts']:
            # Uniformiser base à 80
            base80 = False
            if tokens[0] == 'quatre' and len(tokens) > 1 and tokens[1].startswith('vingt'):
                base80 = True
                rest = tokens[2:]
            elif tokens[0].startswith('quatre-vingt'):
                base80 = True
                rest = tokens[1:]
            if base80:
                if not rest:
                    return 80
                rest_str = '-'.join(rest)
                if rest_str in units:
                    return 80 + units[rest_str]
                if rest_str in teens:
                    return 80 + teens[rest_str]
                if rest_str == 'dix':
                    return 90
                # ex: 'dix-sept' 97, etc.
                if rest_str in teens:  # déjà couvert
                    return 80 + teens[rest_str]
                return pd.NA

        # Cas: composite déjà connu (soixante-dix-sept, etc.)
        if s.startswith('soixante-dix-'):
            tail = s.replace('soixante-dix-', '')
            if tail in units:
                return 70 + units[tail]
            if tail in teens:
                return 70 + teens[tail]
            return pd.NA
        if s.startswith('quatre-vingt-dix-'):
            tail = s.replace('quatre-vingt-dix-', '')
            if tail in units:
                return 90 + units[tail]
            if tail in teens:
                return 90 + teens[tail]
            return pd.NA
    except Exception:
        return pd.NA

    return pd.NA

# Appliquer la conversion
converted = df['age'].apply(fr_text_to_int)

# Rapport
total = len(df)
parsed_text = int(converted.notna().sum() - pd.to_numeric(df['age'], errors='coerce').notna().sum())
invalid = int(converted.isna().sum())

# Assigner au type entier nullable
df['age'] = converted.astype('Int64')

print("\nÉtape 5 – Conversion de 'age' vers entier (avec texte français) :")
print(f"  • Lignes: {total}")
print(f"  • Conversions depuis texte: {parsed_text}")
print(f"  • Valeurs non convertibles (NA): {invalid}")


Étape 5 – Conversion de 'age' vers entier (avec texte français) :
  • Lignes: 420
  • Conversions depuis texte: 37
  • Valeurs non convertibles (NA): 0


In [41]:
# Normaliser 'salaire' : extraire devise et valeur numérique
if 'salaire_raw' not in df.columns:
    df['salaire_raw'] = df['salaire']

import re

def parse_salaire(s):
    if pd.isna(s):
        return pd.Series({'salaire': pd.NA, 'currency': pd.NA})
    src = str(s).strip()
    src_lower = src.lower()
    # Devise
    currency = pd.NA
    if '€' in src or 'eur' in src_lower or 'euro' in src_lower:
        currency = 'euro'
    elif '$' in src or 'usd' in src_lower or 'dollar' in src_lower:
        currency = 'dollar'
    # Nettoyage pour numérique: retirer tout sauf chiffres
    num_str = re.sub(r'[^0-9]', '', src)
    if num_str == '':
        num_val = pd.NA
    else:
        try:
            num_val = int(num_str)
        except Exception:
            num_val = pd.NA
    return pd.Series({'salaire': num_val, 'currency': currency})

parsed = df['salaire'].apply(parse_salaire)
df['salaire'] = parsed['salaire']  # maintenir 'salaire' numérique pour les étapes suivantes
df['currency'] = parsed['currency']
# Rapport salaire
total_sal = len(df)
parsed_ok = int(parsed['salaire'].notna().sum())
missing_cur = int(parsed['currency'].isna().sum())
print("\nÉtape 5 – Normalisation de 'salaire' :")
print(f"  • Lignes: {total_sal}")
print(f"  • Valeurs numériques parsées: {parsed_ok}")
print(f"  • Devise manquante: {missing_cur}")

print("\nAperçu salaire (raw → num | currency):")
print(pd.DataFrame({
    'raw': df['salaire_raw'].head(10),
    'num': df['salaire'].head(10),
    'currency': df['currency'].head(10)
}))



Étape 5 – Normalisation de 'salaire' :
  • Lignes: 420
  • Valeurs numériques parsées: 420
  • Devise manquante: 0

Aperçu salaire (raw → num | currency):
          raw     num currency
0     35 795€   35795     euro
1      96820$   96820   dollar
2  57194euros   57194     euro
3     80263 €   80263     euro
4     84820 €   84820     euro
5     84925 €   84925     euro
6      25311$   25311   dollar
7      73707$   73707   dollar
8     113016$  113016   dollar
9     38431 €   38431     euro


In [42]:
df.head(100)

,id,nom,age,salaire,date_embauche,note_performance,duree_contrat,depart,nom_raw,age_raw,salaire_raw,currency
0,121,david,56,35795,11-Jul-17,5,douze mois,0,David,56,35 795€,euro
1,15,jacques,60,96820,12/4/2019,2,douze mois,1,JACQUES,60,96820$,dollar
2,57,benjamin,36,57194,2019.03.14,3,365 jours,0,Benjamin,36,57194euros,euro
3,204,emmanuel,53,80263,9/15/2020,4,1 an,0,Emmanuel,53,80263 €,euro
4,330,cecile,39,84820,2016.11.24,5,365 jours,0,Cécile,39,84820 €,euro
...,...,...,...,...,...,...,...,...,...,...,...,...
95,168,sabine,25,66214,2018.05.03,4,douze mois,0,SABINE,25,66214euros,euro
96,155,constance,50,90091,5/8/2025,3,douze mois,0,Constance,50,90091euros,euro
97,252,philippine,39,39830,9/6/2016,2,12 mois,1,Philippine,39,39830$,dollar
98,230,dorothee,30,99909,Nov-17,5,douze mois,0,Dorothée,trente,99909euros,euro


In [43]:
# Étape 6 – Corriger le format de la colonne date_embauche (multi-format parser)
import re
from datetime import datetime

def parse_date_multiformat(date_str):
    """
    Parse dates in multiple formats:
    - YYYY.MM.DD (ISO with dots)
    - DD-Mon-YY (e.g., 11-Jul-17)
    - Mon-YY (month-year only, use 1st of month)
    - M/D/YYYY or D/M/YYYY (inferred from value range)
    """
    if pd.isna(date_str):
        return pd.NaT
    
    s = str(date_str).strip()
    
    # Format 1: YYYY.MM.DD (ISO with dots) - unambiguous
    if re.match(r'^\d{4}\.\d{1,2}\.\d{1,2}$', s):
        try:
            parts = s.split('.')
            year, month, day = int(parts[0]), int(parts[1]), int(parts[2])
            return pd.Timestamp(year, month, day)
        except Exception:
            return pd.NaT
    
    # Format 2: DD-Mon-YY or Mon-YY (month abbreviations)
    # Try DD-Mon-YY first
    match = re.match(r'^(\d{1,2})-([A-Za-z]{3})-(\d{2})$', s)
    if match:
        try:
            day, month_str, year_str = match.groups()
            day = int(day)
            year = 2000 + int(year_str) if int(year_str) < 50 else 1900 + int(year_str)
            return pd.to_datetime(f"{day}-{month_str}-{year}", format='%d-%b-%Y')
        except Exception:
            pass
    
    # Try Mon-YY (month-year only)
    match = re.match(r'^([A-Za-z]{3})-(\d{2})$', s)
    if match:
        try:
            month_str, year_str = match.groups()
            year = 2000 + int(year_str) if int(year_str) < 50 else 1900 + int(year_str)
            return pd.to_datetime(f"01-{month_str}-{year}", format='%d-%b-%Y')
        except Exception:
            return pd.NaT
    
    # Format 3: M/D/YYYY or D/M/YYYY (ambiguous)
    # Rule: if first part > 12, it must be day-first; if second part > 12, it must be month-first
    match = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{4})$', s)
    if match:
        try:
            part1, part2, year = int(match.group(1)), int(match.group(2)), int(match.group(3))
            # Heuristic: if part1 > 12, it's day/month/year; else try month/day/year
            if part1 > 12:
                return pd.Timestamp(year, part2, part1)  # D/M/YYYY
            elif part2 > 12:
                return pd.Timestamp(year, part1, part2)  # M/D/YYYY
            else:
                # Both could be month or day; default to M/D/YYYY (common in US)
                return pd.Timestamp(year, part1, part2)
        except Exception:
            return pd.NaT
    
    return pd.NaT

# Preserve raw
if 'date_embauche_raw' not in df.columns:
    df['date_embauche_raw'] = df['date_embauche']

# Apply conversion
df['date_embauche'] = df['date_embauche'].apply(parse_date_multiformat)

# Report
total = len(df)
parsed_ok = int(df['date_embauche'].notna().sum())
invalid = total - parsed_ok

print("\nÉtape 6 – Parsing multi-format de 'date_embauche' :")
print(f"  • Lignes: {total}")
print(f"  • Dates parsées avec succès: {parsed_ok}")
print(f"  • Valeurs non parsées (NA): {invalid}")

# Show distribution by format
formats = {
    'YYYY.MM.DD': 0,
    'DD-Mon-YY': 0,
    'Mon-YY': 0,
    'M/D/YYYY or D/M/YYYY': 0,
    'Unknown/Unparsed': 0
}

for raw_date in df['date_embauche_raw'].dropna():
    s = str(raw_date).strip()
    if re.match(r'^\d{4}\.\d{1,2}\.\d{1,2}$', s):
        formats['YYYY.MM.DD'] += 1
    elif re.match(r'^\d{1,2}-[A-Za-z]{3}-\d{2}$', s):
        formats['DD-Mon-YY'] += 1
    elif re.match(r'^[A-Za-z]{3}-\d{2}$', s):
        formats['Mon-YY'] += 1
    elif re.match(r'^\d{1,2}/\d{1,2}/\d{4}$', s):
        formats['M/D/YYYY or D/M/YYYY'] += 1
    else:
        formats['Unknown/Unparsed'] += 1

print("\nFormat breakdown:")
for fmt, count in sorted(formats.items(), key=lambda x: -x[1]):
    if count > 0:
        print(f"  {fmt}: {count}")

print("\nAperçu (raw → parsed):")
preview = pd.DataFrame({
    'raw': df['date_embauche_raw'].head(15),
    'parsed': df['date_embauche'].head(15)
})
print(preview)



Étape 6 – Parsing multi-format de 'date_embauche' :
  • Lignes: 420
  • Dates parsées avec succès: 420
  • Valeurs non parsées (NA): 0

Format breakdown:
  M/D/YYYY or D/M/YYYY: 203
  YYYY.MM.DD: 82
  DD-Mon-YY: 76
  Mon-YY: 59

Aperçu (raw → parsed):
           raw     parsed
0    11-Jul-17 2017-07-11
1    12/4/2019 2019-12-04
2   2019.03.14 2019-03-14
3    9/15/2020 2020-09-15
4   2016.11.24 2016-11-24
5    20-Jul-15 2015-07-20
6    2/27/2019 2019-02-27
7   11/28/2017 2017-11-28
8   2021.07.05 2021-07-05
9   11/18/2016 2016-11-18
10  17/05/2024 2024-05-17
11   24-Jun-15 2015-06-24
12    7/8/2022 2022-07-08
13      Jun-18 2018-06-01
14   20-Apr-16 2016-04-20


In [44]:
# Étape 7 – Nettoyer la colonne duree_contrat (multi-format parser en mois)

def parse_duration_to_months(val_str):
    """
    Parse contract duration to months:
    - "douze mois" → 12
    - "12 mois" → 12
    - "365 jours" → 12 (approximately)
    - "1 an" / "1 année" / "1 année" → 12
    - "1 an" → 12
    """
    if pd.isna(val_str):
        return pd.NA
    
    s = str(val_str).strip().lower()
    
    # French word mappings for numbers
    french_numbers = {
        'un': 1, 'deux': 2, 'trois': 3, 'quatre': 4, 'cinq': 5,
        'six': 6, 'sept': 7, 'huit': 8, 'neuf': 9, 'dix': 10,
        'onze': 11, 'douze': 12
    }
    
    # Try to extract numeric value first
    match = re.search(r'(\d+)', s)
    if match:
        num = int(match.group(1))
        # Determine the unit
        if 'jour' in s:
            # Convert days to months (365 days ≈ 12 months)
            return round(num / 30.44) if num >= 30 else pd.NA
        elif 'mois' in s:
            return num
        elif 'an' in s or 'année' in s:
            return num * 12
        else:
            # Just a number, assume months
            return num
    
    # Try French text numbers (e.g., "douze mois")
    for word, value in french_numbers.items():
        if word in s:
            if 'jour' in s:
                return round(value / 30.44) if value >= 30 else pd.NA
            elif 'mois' in s:
                return value
            elif 'an' in s or 'année' in s:
                return value * 12
            else:
                return value
    
    return pd.NA

# Preserve raw
if 'duree_contrat_raw' not in df.columns:
    df['duree_contrat_raw'] = df['duree_contrat']

# Apply conversion
df['duree_contrat'] = df['duree_contrat'].apply(parse_duration_to_months)

# Report
total = len(df)
parsed_ok = int(df['duree_contrat'].notna().sum())
invalid = total - parsed_ok

print("\nÉtape 7 – Normalisation de 'duree_contrat' en mois :")
print(f"  • Lignes: {total}")
print(f"  • Valeurs converties en mois: {parsed_ok}")
print(f"  • Valeurs non converties (NA): {invalid}")

# Show distribution by format
formats_count = {}
for raw_val in df['duree_contrat_raw'].dropna():
    s = str(raw_val).strip().lower()
    if re.search(r'\d+', s):
        if 'jour' in s:
            fmt = "Numeric + jours"
        elif 'mois' in s:
            fmt = "Numeric + mois"
        elif 'an' in s:
            fmt = "Numeric + an/année"
        else:
            fmt = "Numeric only"
    else:
        fmt = "Text words + unit"
    formats_count[fmt] = formats_count.get(fmt, 0) + 1

print("\nFormat breakdown:")
for fmt, count in sorted(formats_count.items(), key=lambda x: -x[1]):
    print(f"  {fmt}: {count}")

print("\nAperçu (raw → converted to months):")
preview = pd.DataFrame({
    'raw': df['duree_contrat_raw'].head(20),
    'months': df['duree_contrat'].head(20)
})
print(preview)

# Show summary statistics
print(f"\nStatistiques des durées (en mois):")
print(f"  Min: {df['duree_contrat'].min()}")
print(f"  Max: {df['duree_contrat'].max()}")
print(f"  Moyenne: {df['duree_contrat'].mean():.2f}")
print(f"  Valeurs uniques: {df['duree_contrat'].nunique()}")



Étape 7 – Normalisation de 'duree_contrat' en mois :
  • Lignes: 420
  • Valeurs converties en mois: 420
  • Valeurs non converties (NA): 0

Format breakdown:
  Numeric + an/année: 159
  Text words + unit: 94
  Numeric + mois: 88
  Numeric + jours: 79

Aperçu (raw → converted to months):
           raw  months
0   douze mois      12
1   douze mois      12
2    365 jours      12
3         1 an      12
4    365 jours      12
5   douze mois      12
6      12 mois      12
7    365 jours      12
8         1 an      12
9         1 an      12
10     1 année      12
11  douze mois      12
12     12 mois      12
13     1 année      12
14  douze mois      12
15   365 jours      12
16   365 jours      12
17     12 mois      12
18     12 mois      12
19     1 année      12

Statistiques des durées (en mois):
  Min: 12
  Max: 12
  Moyenne: 12.00
  Valeurs uniques: 1


In [45]:
df.head()

,id,nom,age,salaire,date_embauche,note_performance,duree_contrat,depart,nom_raw,age_raw,salaire_raw,currency,date_embauche_raw,duree_contrat_raw
0,121,david,56,35795,2017-07-11,5,12,0,David,56,35 795€,euro,11-Jul-17,douze mois
1,15,jacques,60,96820,2019-12-04,2,12,1,JACQUES,60,96820$,dollar,12/4/2019,douze mois
2,57,benjamin,36,57194,2019-03-14,3,12,0,Benjamin,36,57194euros,euro,2019.03.14,365 jours
3,204,emmanuel,53,80263,2020-09-15,4,12,0,Emmanuel,53,80263 €,euro,9/15/2020,1 an
4,330,cecile,39,84820,2016-11-24,5,12,0,Cécile,39,84820 €,euro,2016.11.24,365 jours


In [ ]:
# Étape 8 – Standardiser les colonnes numériques et nettoyer les colonnes brutes

# Standardisation (z-score) des colonnes numériques
scaler = StandardScaler()
df[['age_z', 'salaire_z', 'note_performance_z']] = scaler.fit_transform(
    df[['age', 'salaire', 'note_performance']]
)
print("\nÉtape 8 – Aperçu des colonnes standardisées (z-score) :")
df.head()


Étape 8 – Aperçu des colonnes standardisées (z-score) :


,id,nom,age,salaire,date_embauche,note_performance,duree_contrat,depart,currency,age_z,salaire_z,note_performance_z
0,121,david,56,35795,2017-07-11,5,12,0,euro,1.120576,-1.096424,1.834035
1,15,jacques,60,96820,2019-12-04,2,12,1,dollar,1.428789,0.992270,-0.946812
2,57,benjamin,36,57194,2019-03-14,3,12,0,euro,-0.420491,-0.364003,-0.019863
3,204,emmanuel,53,80263,2020-09-15,4,12,0,euro,0.889416,0.425576,0.907086
4,330,cecile,39,84820,2016-11-24,5,12,0,euro,-0.189331,0.581548,1.834035


In [50]:
# Supprimer toutes les colonnes brutes (*_raw) pour garder un jeu de données propre
raw_columns = [col for col in df.columns if col.endswith('_raw')]
if raw_columns:
    print(f"\nSuppression des colonnes brutes : {', '.join(raw_columns)}")
    df = df.drop(columns=raw_columns)
    print(f"Nombre de colonnes restantes : {len(df.columns)}")

print("\nColonnes finales du DataFrame :")
print(df.columns.tolist())


Colonnes finales du DataFrame :
['id', 'nom', 'age', 'salaire', 'date_embauche', 'note_performance', 'duree_contrat', 'depart', 'currency', 'age_z', 'salaire_z', 'note_performance_z']


# Partie B

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Étape 1 – Préparer les données d'entrée
X = df[['age_z', 'salaire_z', 'note_performance_z']]
y = df['depart']

# Étape 2 – Imputation des valeurs manquantes
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Étape 3 – Séparation en données d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.3, random_state=42
)

# Étape 4 – Modèle KNN
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

# Étape 5 – Modèle SVM
svm = SVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

# Étape 6 – Évaluation
acc_knn = accuracy_score(y_test, y_pred_knn)
cm_knn = confusion_matrix(y_test, y_pred_knn)

acc_svm = accuracy_score(y_test, y_pred_svm)
cm_svm = confusion_matrix(y_test, y_pred_svm)

print("Résultats pour KNN :")
print("Accuracy :", acc_knn)
print("Matrice de confusion :")
print(cm_knn)

print("\nRésultats pour SVM :")
print("Accuracy :", acc_svm)
print("Matrice de confusion :")
print(cm_svm)


Résultats pour KNN :
Accuracy : 0.873015873015873
Matrice de confusion :
[[90  6]
 [10 20]]

Résultats pour SVM :
Accuracy : 0.8888888888888888
Matrice de confusion :
[[95  1]
 [13 17]]
